# Interactive Chart Gallery

A compact product telemetry story using pandas, seaborn, matplotlib, and Plotly-generated HTML. The goal is to show how rich notebook outputs remain readable inside a local, sanitized viewer.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import HTML, display

rng = np.random.default_rng(42)
days = pd.date_range("2026-01-01", periods=36, freq="D")
segments = ["research", "finance", "biology"]
rows = []
for segment in segments:
    base = {"research": 62, "finance": 48, "biology": 55}[segment]
    slope = {"research": 0.85, "finance": 0.55, "biology": 0.72}[segment]
    for index, day in enumerate(days):
        sessions = base + slope * index + rng.normal(0, 3.2)
        latency = 220 - 1.4 * index + rng.normal(0, 9) + (8 if segment == "finance" else 0)
        rows.append((day, segment, max(12, sessions), max(90, latency)))

telemetry = pd.DataFrame(rows, columns=["date", "segment", "sessions", "latency_ms"])
summary = telemetry.groupby("segment", as_index=False).agg(
    sessions=("sessions", "mean"),
    latency_ms=("latency_ms", "mean"),
)
summary["sessions"] = summary["sessions"].round(1)
summary["latency_ms"] = summary["latency_ms"].round(1)

display(summary)
print(f"Generated {len(telemetry):,} offline telemetry rows for {len(segments)} segments.")


## Seaborn overview

The viewer treats static chart images as lazy assets, so larger visual sections do not have to be embedded directly into the notebook JSON response.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
sns.lineplot(data=telemetry, x="date", y="sessions", hue="segment", linewidth=2.2, ax=axes[0])
axes[0].set_title("Notebook sessions by audience")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=35)

sns.scatterplot(data=telemetry, x="sessions", y="latency_ms", hue="segment", size="sessions", sizes=(30, 160), alpha=0.75, ax=axes[1])
axes[1].set_title("More reading, lower latency")
axes[1].set_xlabel("daily sessions")
axes[1].set_ylabel("p95 latency (ms)")
fig.tight_layout()
plt.show()


## Plotly HTML and safe fallback

Plotly produces rich HTML and JavaScript. The viewer sanitizes HTML before rendering it, so this cell also includes a visible fallback summary that survives without executing scripts.


In [ ]:
import plotly.express as px

fig = px.line(
    telemetry,
    x="date",
    y="latency_ms",
    color="segment",
    markers=True,
    title="Plotly payload: latency trend by segment",
)
fig.update_layout(template="plotly_white", height=360, margin=dict(l=30, r=20, t=60, b=35))

fallback_rows = "".join(
    f"<tr><td style='padding:7px;border:1px solid #dbe3dc'>{row.segment}</td>"
    f"<td style='padding:7px;border:1px solid #dbe3dc'>{row.sessions:.1f}</td>"
    f"<td style='padding:7px;border:1px solid #dbe3dc'>{row.latency_ms:.1f} ms</td></tr>"
    for row in summary.itertuples(index=False)
)

safe_fallback = f"""
<div style="border:1px solid #dbe3dc;padding:12px;background-color:#fbfcfb;max-width:820px">
  <strong>Sanitized rich-output fallback</strong>
  <p>Scripts are stripped by the viewer, but ordinary HTML remains readable.</p>
  <table style="border-collapse:collapse;width:100%">
    <tr><th style="text-align:left;padding:7px;border:1px solid #dbe3dc">segment</th><th style="text-align:left;padding:7px;border:1px solid #dbe3dc">avg sessions</th><th style="text-align:left;padding:7px;border:1px solid #dbe3dc">avg latency</th></tr>
    {fallback_rows}
  </table>
</div>
"""

display(HTML(safe_fallback))
display(HTML(fig.to_html(include_plotlyjs=False, full_html=False)))
print(f"Plotly figure prepared with {len(fig.data)} traces; the viewer keeps the safe HTML surface.")
